# Autonomous highway driving from camera images (CNN + DQN)

The agent sees **grayscale camera images** of the road (4 stacked frames) instead of a table of car positions, much closer to how a real self-driving car perceives the world.

**Use a GPU:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

Training 50,000 steps with images is much slower than the table version. Expect roughly 30–60+ minutes on a GPU (it varies), and several hours on CPU. On a lab GPU machine you can train longer for better results.

## 1. Get the code and install

In [ ]:
import os
os.chdir("/content")
if not os.path.exists("rl-highway-driving"):
    !git clone https://github.com/bablu2023/rl-highway-driving.git
os.chdir("/content/rl-highway-driving")
!git pull -q
%pip install -q highway-env stable-baselines3 "imageio[ffmpeg]"

import torch
print("GPU available:", torch.cuda.is_available())

## 2. What does the camera see?

In [ ]:
!python show_camera.py
from IPython.display import Image
Image("results/camera_view.png")

## 3. Train the camera-based agent
Same safe settings as the best table-based agent, so the results can be compared.

In [ ]:
!python train.py --obs camera --steps 50000 --gamma 0.95 --collision-reward -3 --out results_camera

In [ ]:
Image("results_camera/training_rewards.png")

## 4. Test it and watch it drive

In [ ]:
!python evaluate.py --model results_camera/dqn_highway.zip --episodes 20 --video results_camera/agent.mp4

In [ ]:
from IPython.display import Video
Video("results_camera/agent.mp4", embed=True)

## 5. Save the results
Download the trained model, plot and a GIF, then add them to the `results/` folder on GitHub (rename to avoid overwriting the table-based results, e.g. `camera_agent.gif`).

In [ ]:
import imageio
from PIL import Image as PILImage
frames = imageio.mimread("results_camera/agent.mp4", memtest=False)
imgs = [PILImage.fromarray(f) for f in frames]
imgs[0].save("results_camera/camera_agent.gif", save_all=True, append_images=imgs[1:], duration=200, loop=0)
!cp results/camera_view.png results_camera/
!cd results_camera && mv training_rewards.png camera_training_rewards.png && mv dqn_highway.zip camera_dqn_highway.zip
!zip -j camera_results.zip results_camera/camera_agent.gif results_camera/camera_training_rewards.png results_camera/camera_view.png results_camera/camera_dqn_highway.zip
from google.colab import files
files.download("camera_results.zip")